[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/internshipinabook/cv-internship-in-a-book/blob/main/notebooks/week1_images_as_arrays_STARTER.ipynb)


# Week 1 -- An Image Is Not a Picture (STARTER)
### The Computer Vision Internship | MediVision AI Lagos

**This week:** pixels as numbers, RGB vs grayscale vs DICOM, CHW format, PIL vs cv2 vs torchvision

**Dataset:** Oxford-IIIT Pets (domain inspection)

**STARTER notebook** -- your working environment. Complete each exercise before moving on.


In [ ]:
# -- Colab/Local Setup -- run this first in Colab, skip locally -------------
import os

if not os.path.exists('requirements.txt'):
    # Clone the full course repository
    !git clone https://github.com/internshipinabook/cv-internship-in-a-book.git cv-book
    %cd cv-book
    # Install all required packages (suppress verbose output with -q)
    !pip install -r requirements.txt -q

# Create working directories
os.makedirs('outputs',  exist_ok=True)   # charts, reports, predictions
os.makedirs('models',   exist_ok=True)   # trained model checkpoints
os.makedirs('datasets', exist_ok=True)   # raw downloaded data
print('Setup complete.')


In [ ]:
# -- Reproducibility Seeds --------------------------------------------------
# Setting seeds ensures every random operation produces the same result
# on every run. Essential for: comparing runs, debugging, sharing results.

import random    # Python built-in random
import numpy as np   # NumPy random (data loading, sklearn)
import torch     # PyTorch random (neural networks, dropout)

SEED = 42        # 42 is conventional -- any integer works

random.seed(SEED)           # fix Python random() calls
np.random.seed(SEED)        # fix np.random.* calls
torch.manual_seed(SEED)     # fix PyTorch CPU operations

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)  # fix GPU operations too

# Prefer deterministic cuDNN algorithms (may slow training ~5%)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Seeds fixed: {SEED} | Device: {device}')


## Learning Objectives

By the end of this week, you will be able to:

- Explain what a pixel is numerically, and why the same colour can have different pixel values across devices
- Distinguish RGB, grayscale, DICOM, and histopathology image formats and their value ranges
- Load an image from each domain (natural, X-ray, microscopy) and describe its tensor properties
- Explain what CHW format means and convert between HWC and CHW without a library function
- Write a one-paragraph description of each dataset for a clinical audience



---

## MONDAY -- "What a Pixel Actually Is"


A pixel in a JPEG cat photo is three uint8 values: Red, Green, Blue, each in [0, 255]. A pixel in a DICOM chest X-ray is one uint16 value in [0, 65535] — representing X-ray attenuation. A pixel in a BreakHis microscopy slide is three uint8 values, but the colour carries diagnostic meaning: haematoxylin stains nuclei blue-purple; eosin stains cytoplasm pink. Knowing what a pixel represents clinically determines how you normalise it.


### Exercise 1.1 -- Pixel vocabulary

For each image type — JPEG pet, DICOM X-ray, BreakHis patch — write: dtype, value range, number of channels, and one sentence explaining what the pixel values represent clinically or photographically.


In [ ]:
# Exercise 1.1: Pixel vocabulary
# -------------------------------------------------------
# YOUR CODE HERE
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the bottom of this notebook.


### Exercise 1.2 -- Shape audit

Load 5 random images from each dataset. Are all images the same size? What is the most common shape? Which dataset has the highest variance in image dimensions?


In [ ]:
# Exercise 1.2: Shape audit
# -------------------------------------------------------
# YOUR CODE HERE
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the bottom of this notebook.


**Monday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Monday: What a Pixel Actually Is (scaffold) --
from PIL import Image
import numpy as np

cat = np.array(Image.open("datasets/pets/Abyssinian_1.jpg"))
print(f"Shape: {cat.shape}, dtype: {cat.dtype}, range: [{cat.min()}, {cat.max()}]")
# Shape: (333, 500, 3), dtype: uint8, range: [0, 255]


### Monday Morning Moment

*Slack — Monday, 11:00am.*

**Yewande Adeyemi:** Ngozi, when are you starting on the CNN?

**Ngozi Eze-Okafor:** Week 5.

**Yewande Adeyemi:** You could get a baseline running today. ResNet-18, fine-tuned on Pets, 20 minutes.

**Ngozi Eze-Okafor:** I don't know what the mean pixel value of a chest X-ray is yet.

**Yewande Adeyemi:** Why does that matter?

**Dr. Kwame Asante:** Yewande — what normalisation parameters did you use for the breast cancer classifier you shipped in 2023?

**Yewande Adeyemi:** ...ImageNet stats.

**Dr. Kwame Asante:** And what happened in the Abuja pilot?

**Yewande Adeyemi:** The model failed on the slides from the university hospital.

**Dr. Kwame Asante:** Ngozi is doing Week 1. Let her do Week 1.




---

## TUESDAY -- "Five Domains, Five Distributions"


Load one image from each dataset. For each, compute: shape, dtype, min, max, mean, std. These five numbers are your domain fingerprint. A model trained on Oxford Pets (mean ≈ [0.48, 0.45, 0.40]) will produce wrong predictions on chest X-rays (mean ≈ [0.50] grayscale) if you apply the same normalisation. The normalisation statistics are domain-specific.


### Exercise 1.3 -- Domain fingerprints

Compute mean and std per channel for 50 random images from each dataset. Build a table: dataset | mean_R | mean_G | mean_B | std_R | std_G | std_B. Which two datasets are most similar? Which are most different?


In [ ]:
# Exercise 1.3: Domain fingerprints
# -------------------------------------------------------
# YOUR CODE HERE
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the bottom of this notebook.


**Tuesday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Tuesday: Five Domains, Five Distributions (scaffold) --
# Domain fingerprint function
def domain_fingerprint(img_path, loader="pil"):
    if loader == "pil":
        arr = np.array(Image.open(img_path).convert("RGB")) / 255.0
    return {
        "shape": arr.shape, "dtype": arr.dtype,
        "mean":  arr.mean(axis=(0,1)).round(4),
        "std":   arr.std(axis=(0,1)).round(4),
        "min":   arr.min(), "max": arr.max()
    }



---

## WEDNESDAY -- "DICOM Windowing — Why Raw Pixels Look Wrong"


A raw DICOM chest X-ray displayed without windowing looks nearly black. The pixel values span [0, 4095] but your monitor displays [0, 255]. Windowing selects a diagnostically relevant range (Window Center + Window Width) and maps it to [0, 255]. Lung windows differ from bone windows. The radiologist's choice of window is a clinical decision, not a display preference.


**Wednesday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Wednesday: DICOM Windowing — Why Raw Pixels Look Wrong (scaffold) --
import pydicom
import numpy as np

ds    = pydicom.dcmread("sample_cxr.dcm")
arr   = ds.pixel_array.astype(np.float32)
WC, WW = 40, 400  # Soft tissue window
lo, hi = WC - WW/2, WC + WW/2
windowed = np.clip((arr - lo) / (hi - lo), 0, 1)



---

## THURSDAY -- "Histopathology — When Colour Is Diagnostic"


In a BreakHis slide, nuclear pleomorphism (irregular, enlarged nuclei stained deep purple) is a malignancy marker. A model that cannot distinguish haematoxylin (blue-purple) from eosin (pink) at the pixel level will fail on this task. Today: load 10 malignant and 10 benign patches. Compute mean colour per class. Is there a visible difference in the colour distribution?


### Exercise 1.4 -- Colour-class relationship

For BreakHis: compute mean RGB separately for benign and malignant patches. Is the difference statistically significant? Run a two-sample t-test on the red channel values. Report: t-statistic, p-value, interpretation.


In [ ]:
# Exercise 1.4: Colour-class relationship
# -------------------------------------------------------
# YOUR CODE HERE
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the bottom of this notebook.


**Thursday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Thursday: Histopathology — When Colour Is Diagnostic (scaffold) --
# Load BreakHis samples
import glob
malignant = glob.glob("datasets/breakhis/malignant/**/*.png", recursive=True)[:10]
benign    = glob.glob("datasets/breakhis/benign/**/*.png",    recursive=True)[:10]

for label, paths in [("Malignant", malignant), ("Benign", benign)]:
    arrs = [np.array(Image.open(p)) / 255.0 for p in paths]
    mean_colour = np.mean([a.mean(axis=(0,1)) for a in arrs], axis=0)
    print(f"{label} mean RGB: {mean_colour.round(3)}")



---

## FRIDAY -- "The Friday Build: Domain Characterisation Notebook"


Your deliverable is a notebook that a clinician could read to understand what each dataset contains. For each of the five datasets: domain fingerprint (shape, dtype, mean, std), class distribution chart, and one clinical sentence explaining why the image looks the way it does — written for Nurse Folake, not for Yewande.


**Friday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Friday: The Friday Build: Domain Characterisation Notebook (scaffold) --
# Each section should answer:
# 1. What does one image look like numerically?
# 2. How are the classes distributed?
# 3. What would a clinician need to know about this image type?


### Friday Workplace Moment

*MediVision AI — Friday standup.*

**Dr. Kwame Asante:** Domain characterisation. Walk me through the most surprising thing you found.

**Ngozi Eze-Okafor:** The BreakHis malignant patches have a lower mean red channel than the benign ones. The malignant nuclei stain darker — more purple — so the red channel intensity drops. That's visible in the pixel statistics before any model sees the data.

**Dr. Kwame Asante:** That is nuclear hyperchromasia. It is a hallmark of malignancy. Write that in the notebook as a clinical annotation.

**Nurse Folake Balogun:** Can I see the colour difference?

**Ngozi Eze-Okafor:** Yes — I made a chart. The malignant mean RGB is [0.71, 0.55, 0.73]. The benign is [0.78, 0.63, 0.79]. The difference is small but consistent across all 10 samples.

**Nurse Folake Balogun:** So the model might learn colour before it learns shape?

**Dr. Kwame Asante:** Write that question down. That is Week 10's GradCAM experiment.



## YOUR WEEK 1 CHECKLIST

Check each item before moving to the next week.
If you cannot explain an item without notes, revisit that section.

- [ ] I can state what a pixel represents in each of the five domains — not just "a number" but what the number means clinically.
- [ ] I have computed domain fingerprints (mean, std) for all five datasets and can explain why they differ.
- [ ] I know what DICOM windowing is and why raw DICOM pixels look wrong without it.
- [ ] I know what nuclear hyperchromasia is and how it appears in pixel statistics.
- [ ] My domain characterisation notebook would make sense to Nurse Folake.


## Challenge Task

> **Core path:** Implement DICOM windowing from scratch (no library function). Apply three different window settings (lung, soft tissue, bone) to the same X-ray. Display all three side by side. Which window would a radiologist choose for pneumonia detection?

> **Advanced path:** Compute the Cohen's d effect size for the malignant vs benign red channel difference in BreakHis. Is the effect clinically meaningful? What sample size would you need to detect it at 80% power?


## Common Mistakes

**Treating all images as uint8 [0,255]:** DICOM is uint16. BreakHis patches may be uint8 but require domain-specific normalisation. Always check dtype before normalising.

**Using Image.open() on a DICOM file:** PIL cannot read DICOM. pydicom.dcmread() is required. The pixel array must be extracted separately from the metadata.

**Skipping the domain fingerprint:** Training without knowing your normalisation target is the single most common cause of silent failure in medical imaging models.



---

## Get the Full Book

This notebook is part of **The Computer Vision Internship** -- Book 3 of 9, InternshipInABook(tm).

Covers: natural images, medical X-rays, breast & uterine cancer, video, GradCAM/LIME/SHAP/Integrated Gradients, and fairness audits.

Available at [internshipinabook.com](https://internshipinabook.com)
